# 05 Window Functions

Use ROW_NUMBER and ranking while keeping partitioning concepts visible.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("module-02-spark-sql").getOrCreate()
base = "../../datasets/module_02"

In [2]:
municipalities = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/municipalities.csv")
accessibility = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/accessibility_scores.csv")
poi = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/poi_counts.csv")
property_values = spark.read.option("header", True).option("inferSchema", True).csv(f"{base}/property_values.csv")
municipalities.printSchema()
municipalities.show(5, truncate=False)

root
 |-- municipality_id: integer (nullable = true)
 |-- municipality_name: string (nullable = true)
 |-- canton: string (nullable = true)
 |-- population: integer (nullable = true)

+---------------+-----------------+------+----------+
|municipality_id|municipality_name|canton|population|
+---------------+-----------------+------+----------+
|1              |Zurich           |ZH    |421878    |
|2              |Winterthur       |ZH    |116122    |
|3              |Uster            |ZH    |36012     |
|4              |Meilen           |ZH    |14733     |
|5              |Bern             |BE    |134591    |
+---------------+-----------------+------+----------+
only showing top 5 rows



In [3]:
municipalities.createOrReplaceTempView("municipalities")
accessibility.createOrReplaceTempView("accessibility_scores")
poi.createOrReplaceTempView("poi_counts")
property_values.createOrReplaceTempView("property_values")

In [9]:
spark.sql("""
SELECT
  municipality_id,
  municipality_name,
  canton,
  population,
  ROW_NUMBER() OVER (
    PARTITION BY canton
    ORDER BY population DESC, municipality_id ASC
  ) AS population_rank_in_canton
FROM municipalities
""").show(30)

+---------------+-----------------+------+----------+-------------------------+
|municipality_id|municipality_name|canton|population|population_rank_in_canton|
+---------------+-----------------+------+----------+-------------------------+
|              5|             Bern|    BE|    134591|                        1|
|              6|      Biel/Bienne|    BE|     55206|                        2|
|              7|             Thun|    BE|     43850|                        3|
|              8|       Interlaken|    BE|      5621|                        4|
|              9|            Basel|    BS|    173552|                        1|
|             10|           Riehen|    BS|     21339|                        2|
|             11|           Geneva|    GE|    203951|                        1|
|             12|          Vernier|    GE|     34898|                        2|
|             13|            Lancy|    GE|     33879|                        3|
|             26|             Chur|    G

In [10]:
spark.sql("""
WITH ranked AS (
  SELECT
    municipality_id,
    municipality_name,
    canton,
    population,
    ROW_NUMBER() OVER (
      PARTITION BY canton
      ORDER BY population DESC, municipality_id ASC
    ) AS population_rank_in_canton
  FROM municipalities
)
SELECT *
FROM ranked
WHERE population_rank_in_canton <= 2
ORDER BY canton, population_rank_in_canton
""").show(30)

+---------------+-----------------+------+----------+-------------------------+
|municipality_id|municipality_name|canton|population|population_rank_in_canton|
+---------------+-----------------+------+----------+-------------------------+
|              5|             Bern|    BE|    134591|                        1|
|              6|      Biel/Bienne|    BE|     55206|                        2|
|              9|            Basel|    BS|    173552|                        1|
|             10|           Riehen|    BS|     21339|                        2|
|             11|           Geneva|    GE|    203951|                        1|
|             12|          Vernier|    GE|     34898|                        2|
|             26|             Chur|    GR|     36783|                        1|
|             27|            Davos|    GR|     10832|                        2|
|             17|          Lucerne|    LU|     82422|                        1|
|             18|            Emmen|    L

In [ ]:
spark.sql("""
SELECT municipality_name, canton, population, accessibility_score,
       ROW_NUMBER() OVER (PARTITION BY canton ORDER BY accessibility_score DESC, population DESC) AS accessibility_rank
FROM municipalities
JOIN accessibility_scores USING (municipality_id)
ORDER BY canton, accessibility_rank
""").show(40, truncate=False)

In [ ]:
spark.sql("""
SELECT * FROM (
  SELECT municipality_name, canton, population,
         ROW_NUMBER() OVER (PARTITION BY canton ORDER BY population DESC) AS population_rank
  FROM municipalities
) ranked
WHERE population_rank <= 2
ORDER BY canton, population_rank
""").show(40, truncate=False)